# 02 — Cleaning and Feature Engineering

**Purpose:** Transform raw channel data into analysis-ready features for partnerships teams.

Features engineered:
- `engagement_proxy` — views per subscriber
- `posting_intensity` — uploads per year of channel age
- `recent_views_momentum` — 30-day views as share of lifetime
- `subscriber_momentum` — 30-day subscriber growth rate
- `momentum_score` — composite recent performance (0–100)
- `consistency_score` — composite stability metric (0–100)
- `earnings_per_1k_views` — estimated monetization efficiency

---

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from ingest_real_data import load_processed_channels
from feature_engineering import engineer_all_features
from utils import set_plot_style, COLORS, TIER_ORDER, format_number

set_plot_style()

In [ ]:
df = load_processed_channels()
print(f'Loaded {len(df)} real channels')

df = engineer_all_features(df)
print(f'Engineered {len([c for c in df.columns if c not in load_processed_channels().columns])} new features')

## 1. Engagement Proxy Distribution

Views per subscriber — the best engagement proxy available at the channel level.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Overall distribution
axes[0].hist(df['engagement_proxy'].dropna().clip(upper=1500), bins=40,
             color=COLORS['primary'], alpha=0.7, edgecolor='white')
axes[0].axvline(df['engagement_proxy'].median(), color=COLORS['danger'], linestyle='--',
                label=f'Median: {df["engagement_proxy"].median():.0f}')
axes[0].set_title('Engagement Proxy (Views/Subscriber)')
axes[0].set_xlabel('Views per Subscriber')
axes[0].legend()

# By tier
tier_data = [df[df['follower_tier'] == t]['engagement_proxy'].dropna() for t in TIER_ORDER if t in df['follower_tier'].values]
tier_labels = [t for t in TIER_ORDER if t in df['follower_tier'].values]
bp = axes[1].boxplot(tier_data, labels=tier_labels, patch_artist=True)
for patch, color in zip(bp['boxes'], [COLORS['primary'], COLORS['secondary'], COLORS['accent'], COLORS['success'], COLORS['danger']]):
    patch.set_facecolor(color)
    patch.set_alpha(0.6)
axes[1].set_title('Engagement Proxy by Tier')
axes[1].set_ylabel('Views per Subscriber')

plt.tight_layout()
plt.show()

## 2. Momentum Features

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Views momentum
mom_data = df['recent_views_momentum'].dropna()
mom_clipped = mom_data[mom_data < mom_data.quantile(0.95)]
axes[0].hist(mom_clipped, bins=40, color=COLORS['secondary'], alpha=0.7, edgecolor='white')
axes[0].set_title('Recent Views Momentum')
axes[0].set_xlabel('30d Views as % of Total')

# Subscriber momentum
sub_mom = df['subscriber_momentum'].dropna()
sub_clipped = sub_mom[sub_mom.between(sub_mom.quantile(0.05), sub_mom.quantile(0.95))]
axes[1].hist(sub_clipped, bins=40, color=COLORS['success'], alpha=0.7, edgecolor='white')
axes[1].set_title('Subscriber Momentum (30d Growth %)')
axes[1].set_xlabel('30d Sub Growth Rate (%)')

# Composite momentum score
axes[2].hist(df['momentum_score'].dropna(), bins=30, color=COLORS['accent'], alpha=0.7, edgecolor='white')
axes[2].set_title('Composite Momentum Score')
axes[2].set_xlabel('Score (0–100)')

plt.tight_layout()
plt.show()

## 3. Posting Intensity and Consistency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Posting intensity
pi = df['posting_intensity'].dropna()
pi_clipped = pi[pi < pi.quantile(0.95)]
axes[0].hist(pi_clipped, bins=40, color=COLORS['primary'], alpha=0.7, edgecolor='white')
axes[0].axvline(pi.median(), color=COLORS['danger'], linestyle='--',
                label=f'Median: {pi.median():.0f} uploads/year')
axes[0].set_title('Posting Intensity (Uploads/Year)')
axes[0].set_xlabel('Uploads per Year')
axes[0].legend()

# Consistency score
axes[1].hist(df['consistency_score'].dropna(), bins=30, color=COLORS['success'], alpha=0.7, edgecolor='white')
axes[1].axvline(df['consistency_score'].median(), color=COLORS['danger'], linestyle='--',
                label=f'Median: {df["consistency_score"].median():.1f}')
axes[1].set_title('Consistency Score (0–100)')
axes[1].set_xlabel('Score')
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Feature Correlation Matrix

In [ ]:
feature_cols = [
    'subscribers', 'total_views', 'uploads', 'engagement_proxy',
    'posting_intensity', 'momentum_score', 'consistency_score',
    'avg_views_per_video', 'views_per_subscriber'
]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df[feature_cols].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, square=True, ax=ax, vmin=-1, vmax=1)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('../dashboard/screenshots/feature_correlation.png')
plt.show()

## 5. Feature Summary by Category

In [ ]:
cat_summary = df.groupby('category').agg(
    channels=('channel_name', 'count'),
    avg_subs=('subscribers', 'mean'),
    avg_engagement=('engagement_proxy', 'mean'),
    avg_momentum=('momentum_score', 'mean'),
    avg_consistency=('consistency_score', 'mean'),
    avg_posting=('posting_intensity', 'mean'),
).sort_values('avg_engagement', ascending=False).round(1)

cat_summary[cat_summary['channels'] >= 5]

---

## Summary

- **Engagement proxy** (views/subscriber) varies significantly by tier and category
- **Momentum scores** reveal which channels are currently trending vs coasting on legacy views
- **Posting intensity** ranges from near-zero to 50K+ uploads/year (network channels)
- **Consistency scores** help identify reliable partners vs volatile channels
- All features derived from real observed metrics — no values fabricated